In [0]:
import sys
import os
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from modules.utils.date import get_target_yyyymm
from modules.data.file_download import download_file

In [0]:
# Download and load to volume bike stations lookup, given a city.
# 1. Construct file paths based on city/system and target month.
# 2. If file already exists, mark job as complete for downstream.
# 3. If not, attempt to download the file from the city's stations URL and upload to target path.
# 4. If successful, mark job as ready for downstream tasks.
months_ago = int(dbutils.widgets.get("months_ago"))
target_month = get_target_yyyymm(months_ago=months_ago)
city = dbutils.widgets.get("city")
city_dict = json.loads(dbutils.widgets.get(city))

file_name: str = f"{city_dict["system"]}_stations_lookup.json"
dir_path: str = f"/Volumes/bikes/00_landing/data_sources/{city_dict["system"]}/stations/{target_month}"
local_path: str = f"{dir_path}/{file_name}"

try:
    dbutils.fs.ls(local_path)

    dbutils.jobs.taskValues.set(key='continue_downstream', value="No")
    print('File already downloaded, aborting downstream tasks')
except:
    try:
        if download_file(url=city_dict["stations_url"],dir_path=dir_path,local_path=local_path):
            dbutils.jobs.taskValues.set(key='continue_downstream', value="Yes")
            print('File succesfully uploaded in current run')
        else:
            dbutils.jobs.taskValues.set(key='continue_downstream', value="No")  
    except Exception as e:
        print(f"Error while downloading file: {str(e)}")
        dbutils.jobs.taskValues.set(key='continue_downstream', value="No")   